# Extra · Publish your project as a website

GitHub already shows your notebooks. With a few extra files, GitHub can also turn your
project into a **website**, like the one this course is published as: your README as the front page, your
notebooks as pages with their results, a sidebar to move between them, and a fixed address you can share,
such as `https://your-name.github.io/your-project/`. Every time you push a change, the website updates
itself.

**Abstract.** This notebook finds the pages of your project (README,
notebooks, documentation), **checks them for the problems that most often break a website**, writes the
settings files and the publishing instructions for GitHub, and, if you want, builds a test version on your
computer. Then you make three clicks on GitHub.

**Before you start, you need:** your project repository on GitHub with at least one notebook (Module 2).

**Run your notebooks and save them before you publish.** The website shows each notebook exactly as it was
last saved, with its results (tables, charts).

Run the cells from top to bottom with **Shift + Enter**.

## Running on Google Colab (run this first)

On your own computer this cell does nothing. On Colab it downloads the course. (To work on *your*
project from Colab, see the configuration below.)

In [ ]:
# --- Google Colab setup (run first) ---
# On Colab this downloads the course so the datasets are available; on your own computer it does nothing.
import os, sys, subprocess                       # standard tools: files, environment, run commands

REPO = "https://github.com/HDSM-TUDarmstadt/open_research_DH.git"   # the course repository
NAME = "open_research_DH"                          # the folder it is downloaded into

if "google.colab" in sys.modules:                  # True only when running on Google Colab
    if os.path.basename(os.getcwd()) != NAME:      # if we are not already inside the course folder
        if not os.path.isdir(NAME):                # and it has not been downloaded yet
            subprocess.run(["git", "clone", REPO], check=True)   # download the course (once)
        os.chdir(NAME)                             # move into it, so the datasets folder is found
    print("Colab ready. Working in:", os.getcwd())
else:                                              # running on your own computer
    print("Running locally: no Colab setup needed.")

## Configuration

Change the values to describe your project. `PROJECT_FOLDER` is the folder of your repository on your
computer (GitHub Desktop puts it in **Documents/GitHub** by default). If that folder does not exist, the
notebook makes a small **demo project** inside the course folder, so you can see what happens first.

**Working only on Colab?** Your project is not on Colab, so put your repository's address in
`COLAB_REPOSITORY` (for example `"https://github.com/your-name/your-project"`; the repository must be
public). The notebook downloads it, prepares the files, and gives you a ZIP to upload on GitHub.

In [ ]:
from pathlib import Path   # a safe way to work with folders and file names

PROJECT_FOLDER = Path.home() / "Documents" / "GitHub" / "open-project-2026"   # your repository's folder

WEBSITE_TITLE = "My open research project"   # shown at the top of every page
AUTHOR        = "Your Name"                  # shown at the bottom of every page
GITHUB_NAME   = "your-github-name"           # your GitHub user name (or your organisation's)
REPOSITORY    = PROJECT_FOLDER.name          # your repository's name on GitHub (usually the folder name)

TEST_BUILD = False         # True: also build a test version of the website on this computer (step 4)
OVERWRITE  = False         # True: replace website files this notebook made earlier (for example after
                           #       you added a new notebook, so it appears on the website)
COLAB_REPOSITORY = ""      # only on Colab: the address of your repository on GitHub

## 1 · Find the pages of your project

The website gets one page for your **README** (the front page), one for each **notebook**, and one for each
Markdown file in your **docs** folder. Folders that hold generated files (`outputs`, `publication`, `data`)
are left out.

In [ ]:
import os, sys, json, shutil, subprocess   # standard tools: folders, settings, running programs

ON_COLAB = "google.colab" in sys.modules                 # True only when running on Google Colab
if ON_COLAB and COLAB_REPOSITORY:                         # Colab: download your project first
    PROJECT_FOLDER = Path("/content") / COLAB_REPOSITORY.rstrip("/").split("/")[-1]
    if not PROJECT_FOLDER.exists():
        subprocess.run(["git", "clone", COLAB_REPOSITORY, str(PROJECT_FOLDER)], check=True)
    REPOSITORY = PROJECT_FOLDER.name

if not PROJECT_FOLDER.exists():                           # no project yet: make a small demo project
    PROJECT_FOLDER = Path("outputs") / "website-demo-project"
    (PROJECT_FOLDER / "notebooks").mkdir(parents=True, exist_ok=True)
    (PROJECT_FOLDER / "docs").mkdir(exist_ok=True)
    if not (PROJECT_FOLDER / "README.md").exists():
        (PROJECT_FOLDER / "README.md").write_text(
            "# Demo project\n\nA small project made by the course, to show how a website is built.\n",
            encoding="utf-8")
        (PROJECT_FOLDER / "docs" / "decision_log.md").write_text(
            "# Decision log\n\n- Used the course dataset, because it is open (CC0).\n", encoding="utf-8")
        import nbformat                                    # the library that reads and writes notebooks
        demo = nbformat.v4.new_notebook()
        demo.metadata = {"kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"},
                         "language_info": {"name": "python"}}
        demo.cells = [nbformat.v4.new_markdown_cell("# Counting records\n\nA first look at the data."),
                      nbformat.v4.new_code_cell("print(2 + 2)")]
        nbformat.write(demo, PROJECT_FOLDER / "notebooks" / "01_counting.ipynb")
    REPOSITORY = PROJECT_FOLDER.name
    print("Your project folder was not found, so this notebook uses a DEMO project:", PROJECT_FOLDER.resolve())
    print("To use your own project, change PROJECT_FOLDER in the configuration cell and run again.\n")

SKIP = {"outputs", "publication", "data", "_build", ".ipynb_checkpoints", ".git", ".github"}

def project_files(pattern):
    """All files matching the pattern in the project, except those in the skipped folders, sorted."""
    found = []
    for f in sorted(PROJECT_FOLDER.rglob(pattern)):
        parts = f.relative_to(PROJECT_FOLDER).parts
        if not any(p in SKIP or p.startswith(".") for p in parts):
            found.append(f)
    return found

readme    = PROJECT_FOLDER / "README.md"
notebooks = project_files("*.ipynb")
docs      = [f for f in project_files("*.md") if f.relative_to(PROJECT_FOLDER).parts[0] == "docs"]

print("Project folder:", PROJECT_FOLDER.resolve())
print("Front page:    ", "README.md" if readme.exists() else "MISSING (see step 2)")
print(f"Notebooks ({len(notebooks)}):", *[" " + str(f.relative_to(PROJECT_FOLDER)) for f in notebooks], sep="\n")
print(f"Documentation pages ({len(docs)}):", *[" " + str(f.relative_to(PROJECT_FOLDER)) for f in docs], sep="\n")

## 2 · Check the pages

These are the problems that most often break a website or make it look wrong. The cell explains each
problem it finds and how to fix it. Fix them, save, and run this cell again until it says *No problems
found*. (Warnings are fine to publish with, but the website looks better without them.)

In [ ]:
import nbformat                     # the library that reads and writes notebooks

problems, warnings = [], []
damaged = []                                               # notebooks that cannot be read are left out

if not readme.exists():                                    # the website needs a front page
    readme.write_text(f"# {WEBSITE_TITLE}\n\nWhat this project is about, in two or three sentences.\n",
                      encoding="utf-8")
    warnings.append("There was no README.md, so a short one was created. Open it and describe your project.")

def first_heading(lines):
    """Return the first Markdown heading line, or None."""
    for line in lines:
        if line.startswith("#"):
            return line
    return None

for f in notebooks:
    name = f.relative_to(PROJECT_FOLDER)
    try:
        book = nbformat.read(f, as_version=4)                # can the notebook be opened at all?
    except Exception as err:
        damaged.append(f)
        problems.append(f"{name}: this notebook file is damaged and cannot be read ({type(err).__name__}). "
                        "It is left out of the website. Open it in Jupyter; if it does not open, restore "
                        "it from GitHub Desktop (History) or remove it from the project.")
        continue
    md_lines = [l for c in book.cells if c.cell_type == "markdown" for l in c.source.splitlines()]
    head = first_heading(md_lines)
    if head is None or not head.startswith("# "):
        problems.append(f"{name}: has no title. Add a text cell at the very top with one line starting with "
                        "'# ' (a hash and a space), for example '# Counting records'. The title becomes the "
                        "page name in the website's sidebar.")
    if not any(c.get("outputs") for c in book.cells if c.cell_type == "code"):
        warnings.append(f"{name}: shows no results yet. Run it (Kernel > Restart & Run All) and save it, so "
                        "the website shows its tables and charts.")
    if " " in f.name:
        warnings.append(f"{name}: the file name contains spaces, so its web address will contain '%20'. "
                        "Consider renaming it with underscores (for example 01_counting.ipynb).")

for f in [readme] + docs:
    lines = f.read_text(encoding="utf-8", errors="replace").splitlines()
    head = first_heading(lines)
    if head is None or not head.startswith("# "):
        problems.append(f"{f.relative_to(PROJECT_FOLDER)}: has no title. Make the first heading a line "
                        "starting with '# ' (one hash and a space).")

for f in PROJECT_FOLDER.rglob("*"):                         # GitHub refuses files over 100 MB
    if f.is_file() and ".git" not in f.parts and f.stat().st_size > 50 * 1024 * 1024:
        problems.append(f"{f.relative_to(PROJECT_FOLDER)} is {f.stat().st_size // 2**20} MB. GitHub refuses "
                        "files over 100 MB and warns above 50 MB. Keep large data out of the repository "
                        "(put it on Zenodo, Module 8) and add its folder to .gitignore.")

if not notebooks:
    warnings.append("There are no notebooks yet. The website will show only your README and docs.")

for p in problems:
    print("PROBLEM:", p, "\n")
for w in warnings:
    print("Warning:", w, "\n")
if not problems:
    print("No problems found." + (" (Only warnings, see above.)" if warnings else ""))

## 3 · Write the website files

The cell writes these files into your project. You do not need to edit them:

| File | What it is for |
|---|---|
| `_config.yml` | the website's settings: title, author, the link to your repository |
| `_toc.yml` | the order of the pages in the sidebar: README first, then the notebooks, then the docs |
| `requirements-website.txt` | what GitHub installs to build the website |
| `.github/workflows/publish-website.yml` | the instructions GitHub follows to build and publish the website |
| `_static/custom.css` | the colours and fonts of this course (delete it if you prefer the plain look) |

Files that already exist are kept, unless you set `OVERWRITE = True`. Do that after you **add, rename, or
delete a notebook**, so the sidebar matches your project again.

In [ ]:
def page(f):
    """The name of a page in _toc.yml: its path inside the project, without the file extension."""
    return f.relative_to(PROJECT_FOLDER).with_suffix("").as_posix()

notebooks = [f for f in notebooks if f not in damaged]  # damaged notebooks would stop the website

toc = ["# The order of the pages on your website (made by the course notebook)",
       "format: jb-book", "root: README", "parts:"]
if notebooks:
    toc += ["  - caption: Notebooks", "    chapters:"] + [f"      - file: {page(f)}" for f in notebooks]
if docs:
    toc += ["  - caption: Documentation", "    chapters:"] + [f"      - file: {page(f)}" for f in docs]
if not notebooks and not docs:
    toc = toc[:-1]                                         # a website with only the front page

config = f"""# Your website's settings (made by the course notebook)
title: {json.dumps(WEBSITE_TITLE)}
author: {json.dumps(AUTHOR)}
only_build_toc_files: true          # only the pages listed in _toc.yml become part of the website

execute:
  execute_notebooks: "off"          # show each notebook as you saved it, with its results

repository:
  url: https://github.com/{GITHUB_NAME}/{REPOSITORY}
  branch: main

launch_buttons:
  colab_url: "https://colab.research.google.com"   # a button that opens each notebook in Google Colab

html:
  use_repository_button: true
  use_download_button: true

parse:
  myst_enable_extensions: [colon_fence, deflist, tasklist, html_image]
"""

WORKFLOW = """# Publishes the website on GitHub Pages every time you push to the main branch.
# You do not need to change anything in this file.
# Once per repository: Settings > Pages > Source: "GitHub Actions".
name: Publish the project website

on:
  push:
    branches: [main, master]
  workflow_dispatch:            # adds a "Run workflow" button in the Actions tab

permissions:
  contents: read
  pages: write
  id-token: write

concurrency:
  group: pages
  cancel-in-progress: false

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with:
          python-version: "3.11"
      - name: Install Jupyter Book
        run: pip install -r requirements-website.txt
      - name: Build the website
        run: jupyter-book build . --keep-going
      - uses: actions/upload-pages-artifact@v4
        with:
          path: _build/html

  deploy:
    needs: build
    runs-on: ubuntu-latest
    environment:
      name: github-pages
      url: ${{ steps.deployment.outputs.page_url }}
    steps:
      - id: deployment
        uses: actions/deploy-pages@v4
"""

files = {"_toc.yml": "\n".join(toc) + "\n",
         "_config.yml": config,
         "requirements-website.txt": "jupyter-book==1.0.4.post1\n",
         ".github/workflows/publish-website.yml": WORKFLOW}

written = []
for name, text in files.items():
    target = PROJECT_FOLDER / name
    if target.exists() and not OVERWRITE:
        print("kept (already there):", name)
        continue
    target.parent.mkdir(parents=True, exist_ok=True)       # makes .github/workflows if needed
    target.write_text(text, encoding="utf-8")
    written.append(name)
    print("written:", name)

for style in [Path("../_static/custom.css"), Path("_static/custom.css")]:   # the course's colours and fonts
    target = PROJECT_FOLDER / "_static" / "custom.css"
    if style.exists() and (OVERWRITE or not target.exists()):
        target.parent.mkdir(exist_ok=True)
        shutil.copy(style, target)
        print("written: _static/custom.css (the course's colours; delete it for the plain look)")
        break

gitignore = PROJECT_FOLDER / ".gitignore"                  # the test build should not go to GitHub
old = gitignore.read_text(encoding="utf-8") if gitignore.exists() else ""
if "_build" not in old:
    gitignore.write_text(old.rstrip("\n") + "\n# website test build\n_build/\n", encoding="utf-8")
    print("added _build/ to .gitignore")

## 4 · Optional: build a test version on your computer

GitHub builds the website for you, so this step is optional. It is useful when you want to **see the
website before you publish it**, or when GitHub reports an error and you want to find it. Set
`TEST_BUILD = True` in the configuration cell, run it, and run the cell below. The first time, it
installs Jupyter Book (about a minute). On Colab, the test version cannot be opened, so skip this step there.

In [ ]:
import importlib.util                     # to check whether a library is installed

def build_website(folder):
    """Build the website with Jupyter Book and return its messages."""
    code = ("import sys; from jupyter_book.cli.main import main; "
            f"sys.argv = ['jupyter-book', 'build', {str(folder)!r}]; main()")
    run = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return run.returncode, run.stdout + run.stderr

EXPLAIN = {                                            # common messages, in plain words
    "toctree contains reference to nonexisting document":
        "_toc.yml lists a page that no longer exists (renamed or deleted?). Run step 3 with OVERWRITE = True.",
    "Non-consecutive header level increase":
        "a heading level is skipped (for example # followed by ###). Use ## before ###.",
    "image file not readable":
        "an image is linked with a path that does not exist. Check the file name and folder.",
    "cross-reference target not found":
        "a link points to a file that is not part of your website (for example a course page). Change the "
        "link to a full web address, or remove it.",
    "No source code lexer found":
        "the notebook does not say which language it uses. Open it in Jupyter, choose the Python 3 kernel, "
        "and save it.",
    "Document headings start at":
        "a page starts with ## instead of # . Give it a title line starting with '# '.",
}

if not TEST_BUILD:
    print("Skipped: set TEST_BUILD = True in the configuration cell to build a test version.")
else:
    if importlib.util.find_spec("jupyter_book") is None:      # install Jupyter Book the first time
        print("Installing Jupyter Book (once)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyter-book==1.0.4.post1"], check=True)
    code, log = build_website(PROJECT_FOLDER)
    import re
    log = re.sub(r"\[[0-9;]*m", "", log)                      # remove colour codes
    log = log.replace(str(PROJECT_FOLDER.resolve()) + os.sep, "")  # show paths inside the project only
    messages = [l for l in log.splitlines() if "WARNING" in l or "ERROR" in l]
    for line in messages:
        hint = next((v for k, v in EXPLAIN.items() if k in line), "")
        print("-", line.strip()[:200], ("\n  What to do: " + hint) if hint else "")
    index = PROJECT_FOLDER / "_build" / "html" / "index.html"
    if code == 0 and index.exists():
        print("\nThe website was built" + (" without problems." if not messages else ", with the messages above."))
        print("Open this file in your browser to look at it:", index.resolve())
    else:
        causes = [l for l in log.splitlines()                  # the lines that name the cause
                  if ("Error" in l or "Exception" in l) and not l.startswith(" ") and "RuntimeError" not in l]
        print("\nThe build failed. Fix the problems from step 2 first. The cause reported by Jupyter Book:")
        print("\n".join(causes[-5:]) if causes else "\n".join(log.splitlines()[-15:]))

## 5 · Publish: three clicks on GitHub

1. **Switch on GitHub Pages (once).** On GitHub, open your repository, click **Settings** (the gear icon at
 the top), then **Pages** in the left menu. Under **Build and deployment**, set **Source** to
 **GitHub Actions**. That is all; there is nothing else to fill in.
2. **Commit and push.** In GitHub Desktop you now see the new files (`_config.yml`, `_toc.yml`,
 `requirements-website.txt`, and `.github/workflows/publish-website.yml`). Write a summary such as
 `Publish the project as a website`, click **Commit to main**, then **Push origin**.
3. **Wait for the green tick.** On GitHub, open the **Actions** tab. A yellow dot means GitHub is building
 the website (2 to 4 minutes); a **green tick** means it is online.

Your website's address is printed by the cell below. Add it to the top of your README, so that visitors
find it. From now on, **every push updates the website**.

**Free GitHub Pages needs a public repository.** For a private repository you need GitHub Pro, which is
free for students through [GitHub Education](https://education.github.com).

In [ ]:
address = f"https://{GITHUB_NAME.lower()}.github.io/{REPOSITORY}/"
print("Your website will be at:", address)
print("Add this line near the top of your README.md:")
print(f"\n**Website:** <{address}>\n")

if ON_COLAB:                                      # Colab: pack the files to upload them on GitHub
    import zipfile
    zip_path = Path("/content/website_files.zip")
    with zipfile.ZipFile(zip_path, "w") as z:
        for name in ["_config.yml", "_toc.yml", "requirements-website.txt"]:
            z.write(PROJECT_FOLDER / name, name)
    print("On Colab: download", zip_path.name, "from the Files panel, unzip it, and upload the three files on")
    print("GitHub (your repository > Add file > Upload files). Then click Add file > Create new file, type the")
    print("name .github/workflows/publish-website.yml and paste this text into it:\n")
    print(files[".github/workflows/publish-website.yml"])

## If something goes wrong

| What you see | What to do |
|---|---|
| A **red cross** in the Actions tab | Click it, then click **build** or **deploy** to read the message. Run step 2 of this notebook again: it finds most causes. |
| The message *Get Pages site failed* or *Not Found* | GitHub Pages is not switched on: do step 5.1, then in the Actions tab click the failed run and **Re-run all jobs**. |
| The address shows **404** | Wait a few minutes after the first green tick. Check that the repository is public and that the address uses your user name and repository name. |
| A notebook is **missing** from the sidebar | It was added after the website files were written: run step 3 with `OVERWRITE = True`, then commit and push. |
| A notebook shows **code but no results** | Run it in Jupyter, save it, commit, and push. |
| The website looks right but is **out of date** | Check that you pushed (not only committed) in GitHub Desktop, and wait for the green tick. |

---
**Back to:** [Course wrap-up](09_wrap_up.ipynb)